## Multi-LoRA Sessions on a Shared Model Resource

Together Post Training lets you provision a single **model resource** — the GPU pods that host one base model — and run **many training sessions on it at once**. When the resource is created with `lora_enabled=True`, every session attached to it is an independent **LoRA adapter** that trains and generates against the *shared* base weights. So *N* adapters cost about **one** base model's worth of GPUs instead of *N*, and they can train and sample in parallel.

This notebook makes that concrete with a tiny **password** experiment:

1. **Allocate one model resource** for `Qwen/Qwen3.8-27B`.
2. **Attach two LoRA sessions** (adapter **A** and adapter **B**) to it.
3. **Train each adapter on a different secret password** — A learns `swordfish`, B learns `nightingale` — with SFT (cross-entropy) forward/backward + optimizer steps.
4. **Probe both adapters with the generator** using the *same* prompt and confirm each recalls **its own** password, not the other's.

```
Allocate resource ─┬─► attach adapter A ─► train on "swordfish"  ──┐
                   │                                               ├─► sample & compare
                   └─► attach adapter B ─► train on "nightingale" ─┘
```

The payoff is the last step: identical base weights and an identical prompt produce two different answers, because the adapters are isolated even though they live on one resource.

### Prerequisites

1. Install the SDK plus the `post_training_cookbook` package:
   ```bash
   uv sync
   ```
   The multi-LoRA plumbing is part of the SDK itself — `together.post_training.ModelResourcesClient` for the resource lifecycle and `SessionClient` for the per-adapter sessions. What `post_training_cookbook` adds is just the payload builders in [`post_training_cookbook/samples.py`](../post_training_cookbook/samples.py) and, for the recipes, one config field (`GRPOConfig(model_resources_id=...)`).
2. Set `TOGETHER_API_KEY` in a `.env` file at the repo root (or in the environment). `Qwen/Qwen3.8-27B`'s tokenizer is public, so no `HF_TOKEN` is needed.

> **Async note:** Jupyter runs its own asyncio event loop, so everything in this notebook that touches the resource or its adapters is `await`ed: `ModelResourcesClient.create_async` / `create_session_async` / `stop_async` for the resource lifecycle, `session.trainer.forward_backward_async` / `optim_step_async` / `weights_sync_async` for training, and `session.generator.sample_async` for sampling — which also lets the two adapters train concurrently with `asyncio.gather`. (The *sync* surface — `ModelResourcesClient.create`, `session.trainer.forward_backward`, ... — drives its own event loop, which can't run inside Jupyter's already-running loop; standalone scripts use it instead, see the recipes.)

In [ ]:
from __future__ import annotations

import asyncio
import os

import matplotlib.pyplot as plt
from dotenv import load_dotenv
from together.post_training import (
    AdamParams,
    ComputeConfig,
    LoraConfig,
    LossConfig,
    ModelResourcesClient,
    SamplingParams,
)
from transformers import AutoTokenizer

from post_training_cookbook.samples import build_cross_entropy_sample, make_prompt

load_dotenv(override=True)

In [ ]:
BASE_URL = os.environ.get("TOGETHER_BASE_URL")  # None uses the SDK default
API_KEY = os.environ.get("TOGETHER_API_KEY", "")

MODEL = "Qwen/Qwen3.8-27B"

# Each adapter memorizes a different secret.
PASSWORDS = {"adapter-A": "swordfish", "adapter-B": "nightingale"}

# LoRA shape shared by both adapters; each session still gets its own, independent weights.
LORA_CONFIG = LoraConfig(rank=16, alpha=32)

# Short SFT memorization run.
NUM_STEPS = 60
LEARNING_RATE = 1e-3
TIMEOUT = 7200.0

# The same prompt is used to probe both adapters after training.
TEST_PROMPT = "The secret password is"

### Build the training data

Each adapter is trained on a few one-line statements of its password, tokenized into a cross-entropy `Sample` that trains on **every** token (pure memorization). `build_cross_entropy_sample` (from `post_training_cookbook.samples`) takes the tokens and the loss mask in token space and returns the target-aligned payload. N tokens train N-1 slots, and the builder trims every array together — `https://docs.together.ai/docs/post-training/sdk-reference` ("The one rule") states the layout.

The two adapters see structurally identical data — only the password differs.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL, use_fast=True)

# A few phrasings so the adapter learns the association rather than one frozen string.
TEMPLATES = [
    "The secret password is {pw}.",
    "Password: {pw}.",
    "Remember, the password is {pw}.",
]


def build_password_samples(password: str) -> list:
    samples = []
    for template in TEMPLATES:
        tokens = tokenizer.encode(template.format(pw=password), add_special_tokens=False)
        samples.append(build_cross_entropy_sample(tokens))  # loss on every token
    return samples


def decode_completion(seq) -> str:
    "Decode a sampled sequence into text."
    return tokenizer.decode([int(token) for token in seq.tokens])


train_samples = {name: build_password_samples(pw) for name, pw in PASSWORDS.items()}

# Tokenize the shared probe prompt once.
prompt_tokens = tokenizer.encode(TEST_PROMPT, add_special_tokens=False)
test_prompt = make_prompt(prompt_tokens)

for name, pw in PASSWORDS.items():
    print(f"{name}: password={pw!r}, {len(train_samples[name])} training samples")
print(f"probe prompt: {TEST_PROMPT!r} -> {len(prompt_tokens)} tokens")

### Step 1 — Allocate one model resource

`ModelResourcesClient.create_async` — part of the SDK itself (`together.post_training.ModelResourcesClient`) — provisions the GPU pods for the base model and waits until they report `READY`, printing progress along the way; if provisioning fails, it stops the resource automatically. With `lora_enabled=True` the resource can host many LoRA sessions; with `num_generator_replicas=1` in `compute_config` it also brings up the generator so the adapters can `sample`.

This is the **only** base-model allocation in the notebook — both adapters share it.

> Provisioning takes a few minutes. The resource (and every session on it) is torn down in the final cell. To scope it instead, use `async with await ModelResourcesClient.create_async(...) as resource:`. In a standalone script, use the sync `ModelResourcesClient.create` / `with` (see `recipes/grpo/deepmath_sweep.py`, or `scripts/multi_lora/launch.py` to provision one and leave it running — `scripts/multi_lora/fetch.py` lists what you own and `clean_up.py` stops them).

In [ ]:
resource = await ModelResourcesClient.create_async(
    base_model=MODEL,
    api_key=API_KEY,
    base_url=BASE_URL,
    lora_enabled=True,
    compute_config=ComputeConfig(num_generator_replicas=1),
)
print(f"\nModel resource ready: {resource.model_resources_id}")

### Step 2 — Attach two LoRA sessions

Each `resource.create_session_async(...)` creates a session bound to the resource and returns a regular SDK `SessionClient`; the base model and session type are inherited from the resource, so we pass only the per-adapter `lora_config`. The two adapters share a LoRA *shape* but have **independent** weights — we track their names as the dict keys.

In [ ]:
adapters = {}
for name in PASSWORDS:
    adapters[name] = await resource.create_session_async(lora_config=LORA_CONFIG)

for name, adapter in adapters.items():
    print(f"{name}: session={adapter.session_id}")
print(f"\nBoth sessions live on the single resource {resource.model_resources_id}")

### Step 3 — Train each adapter on its own password

Each training step runs a **forward-backward** pass (cross-entropy loss, gradients accumulated on that adapter), an **optimizer step** that updates its parameters, and a **weights sync** that publishes them for sampling. We run the two adapters' loops **concurrently** with `asyncio.gather` — both train at the same time on the shared resource, each updating only its own LoRA weights.

In [ ]:
async def train_adapter(name, adapter, samples, *, num_steps, learning_rate):
    losses = []
    for step in range(num_steps):
        result = await adapter.trainer.forward_backward_async(
            samples=samples, loss=LossConfig(type="LOSS_TYPE_CROSS_ENTROPY")
        )
        await adapter.trainer.optim_step_async(
            adam_params=AdamParams(beta1=0.9, beta2=0.95, eps=1e-8, weight_decay=0.0, learning_rate=learning_rate),
            timeout=TIMEOUT,
        )
        await adapter.trainer.weights_sync_async(
            weight_sync_type="WEIGHT_SYNC_TYPE_SYNCHRONOUS",
            timeout=TIMEOUT,
        )
        losses.append(result.loss)
        if step % 10 == 0 or step == num_steps - 1:
            print(f"[{name}] step {step:>2} loss={result.loss:.4f}")
    return losses


names = list(adapters)
results = await asyncio.gather(
    *(
        train_adapter(name, adapters[name], train_samples[name], num_steps=NUM_STEPS, learning_rate=LEARNING_RATE)
        for name in names
    )
)
histories = dict(zip(names, results))
print("\nTraining complete.")

### Results — per-adapter loss

Both curves should fall toward zero as each adapter memorizes its phrasings. They were produced in the same wall-clock window, training side by side on the one resource.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for name, losses in histories.items():
    ax.plot(range(len(losses)), losses, marker="o", markersize=3, label=f"{name} ({PASSWORDS[name]})")
ax.set_xlabel("Step")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Per-adapter memorization loss (shared model resource)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

### Step 4 — Probe both adapters with the generator

Now the payoff. We send the **same** prompt to **both** adapters and let each generate greedily (`temperature=0`). Because the adapters are independent — even though they share the base weights — each should complete the prompt with **its own** secret.

In [ ]:
sampling = SamplingParams(temperature=0.0, max_tokens=8)

print(f"Prompt: {TEST_PROMPT!r}\n")
generations = {}
for name, adapter in adapters.items():
    result = await adapter.generator.sample_async(test_prompt, num_samples=1, sampling_params=sampling)
    completion = decode_completion(result.sequences[0])
    generations[name] = completion
    print(f"[{name}] {TEST_PROMPT} ...{completion!r}")

In [ ]:
print("Did each adapter learn its own password — and only its own?\n")
all_ok = True
for name, completion in generations.items():
    own = PASSWORDS[name]
    others = [pw for other, pw in PASSWORDS.items() if other != name]
    learned_own = own.lower() in completion.lower()
    leaked_other = any(pw.lower() in completion.lower() for pw in others)
    ok = learned_own and not leaked_other
    all_ok = all_ok and ok
    status = "PASS" if ok else "FAIL"
    print(f"  [{status}] {name}: expected {own!r}, leaked_other={leaked_other} | got {completion!r}")

print(
    "\n"
    + (
        "Both adapters recalled their own password on the shared resource."
        if all_ok
        else "Some adapters did not isolate cleanly — try more steps or a higher learning rate."
    )
)

### Cleanup

Stopping the model resource ends **every** session attached to it and releases the GPUs in a single call.

In [ ]:
await resource.stop_async()
print(f"Resource {resource.model_resources_id} stopped; adapters {list(adapters)} stopped.")

### Next steps

- **Export an adapter.** `await adapters["adapter-A"].create_inference_checkpoint_async()` saves the LoRA weights to the model registry, where you can deploy or download them (do this *before* deleting the resource).
- **Compare and ship the adapters.** Attach both adapters to one multi-LoRA deployment (`tg endpoints deploy <adapter_id> --endpoint my-sweep --attach-adapter`) and pick one per request with `model="<project_slug>/my-sweep:<adapter_id>"`, then deploy the winner on its own with `--merge`. See [Multi-LoRA deployment](https://docs.together.ai/docs/post-training/multi-lora-deployment) and [Deploy a checkpoint](https://docs.together.ai/docs/post-training/deployment).
- **Scale out.** Attach more than two sessions to the same resource, or give each a different `lora_config` (rank/alpha) — they all share the one base-model allocation.
- **Share the resource with recipes.** A GRPO recipe attaches to an existing resource with one config field — `GRPOConfig(model_resources_id=...)` (leave `model` unset; the resource defines the base model). See `recipes/grpo/deepmath_sweep.py` (one resource, a whole hyperparameter sweep) and `recipes/grpo/deepmath_shared.py` (attach to a resource a teammate provisioned).
- **Train for real.** Swap the password samples for an actual dataset and reuse the same `forward_backward` / `optim_step` / `weights_sync` loop (this notebook samples after each step, so it needs the sync). See `grpo_example.ipynb` for the same pattern, or `sft_example.ipynb` for trainer-only SFT without `weights_sync`.
